# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('tic-tac-toe.csv')

if set(df.columns[:9].str.lower()) <={'x','o','b'}:
    cols = ['TL', 'TM', 'TR', 'ML', 'MM', 'MR', 'BL', 'BM', 'BR', 'class']
    df = pd.read_csv('tic-tac-toe.csv', header=None,names=cols)
print(df.shape)
print(df.head())
print(df.dtypes)
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

board_cols = df.columns[:-1]
target_col = df.columns[-1]

for c in df.columns:
    print(c, df[c].unique())              
print(df[target_col].value_counts())       

n_x = (df[board_cols] == 'x').sum(axis=1)
n_o = (df[board_cols] == 'o').sum(axis=1)
valid = ((n_x - n_o).isin([0, 1]))
print("Boards with a legal number of moves:", valid.sum(), "/", len(df))

square_map = {'x': 1, 'o': -1, 'b': 0}
df[board_cols] = df[board_cols].apply(lambda s: s.str.lower().map(square_map))

df[target_col] = (df[target_col].astype(str).str.lower()
                  .map({'positive': 1, 'true': 1, 'negative': 0, 'false': 0}))
print(df.head())

X = df[board_cols].values.astype(float)
y = df[target_col].values

X = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

print("X shape:", X.shape, " y shape:", y.shape)
print("X range:", X.min(), "to", X.max())

(958, 10)
  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True
TL        str
TM        str
TR        str
ML        str
MM        str
MR        str
BL        str
BM        str
BR        str
class    bool
dtype: object
Missing values:
 TL       0
TM       0
TR       0
ML       0
MM       0
MR       0
BL       0
BM       0
BR       0
class    0
dtype: int64
Duplicate rows: 0
TL <StringArray>
['x', 'o', 'b']
Length: 3, dtype: str
TM <StringArray>
['x', 'o', 'b']
Length: 3, dtype: str
TR <StringArray>
['x', 'o', 'b']
Length: 3, dtype: str
ML <StringArray>
['x', 'o', 'b']
Length: 3, dtype: str
MM <StringArray>
['o', 'b', 'x']
Length: 3, dtype: str
MR <StringArray>
['o', 'b', 'x']
Length: 3, dtype: str
BL <StringArray>
['x', 'o', 'b']
Length: 3, dtype: str
BM <StringArray>
['o', 'x', 'b']
Length: 3, dtype: str
BR <StringArray>
['o',

## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [4]:
pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 10.5 MB/s  0:00:52m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 12.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 7.1 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 10.8 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 13.2 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/24 [tensorflow]4 [tensorflow]-py]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# 1. Split the training and test data (80 / 20, keeping the class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)

# 2. Create a Sequential model
model = tf.keras.models.Sequential()

# 3. Add layers: ReLU in the middle, Softmax at the output
model.add(tf.keras.layers.Input(shape=(X_train.shape[1],)))     # 9 board squares
model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dense(2, activation='softmax'))       # 2 classes: negative / positive

# 4. Compile
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

# 5. Fit the training data
history = model.fit(X_train, y_train, epochs=50, batch_size=32,
                    validation_split=0.1, verbose=1)

# 6. Evaluate with the test data
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}   Test accuracy: {test_acc:.4f}")

# 7. Save the model
try:
    model.save('tic-tac-toe.model')          # TensorFlow < 2.16 (Keras 2): SavedModel folder
except ValueError:
    model.save('tic-tac-toe.model.keras')    # TensorFlow >= 2.16 (Keras 3) requires an extension

I0000 00:00:1795238511.530761     901 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795238511.531331     901 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795238511.724065     901 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795238512.781561     901 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

Train: (766, 9)  Test: (192, 9)


E0000 00:00:1795238514.112443     901 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,178 (102.26 KB)

 Trainable params: 26,178 (102.26 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6386 - loss: 0.6211 - val_accuracy: 0.6623 - val_loss: 0.5974
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7373 - loss: 0.5328 - val_accuracy: 0.6883 - val_loss: 0.5664
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7562 - loss: 0.4921 - val_accuracy: 0.7403 - val_loss: 0.5334
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8142 - loss: 0.4365 - val_accuracy: 0.7922 - val_loss: 0.5010
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8534 - loss: 0.3648 - val_accuracy: 0.8052 - val_loss: 0.4211
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9086 - loss: 0.2860 - val_accuracy: 0.8701 - val_loss: 0.3140
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9608 - loss: 0.1949 - val_accuracy: 0.9221 - val_loss: 0.2281
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9782 - loss: 0.1285 - val_accuracy: 1.0000 - val_loss:

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [6]:
import os
import numpy as np
import tensorflow as tf

# Load the saved model (works with either name from Step 2)
path = 'tic-tac-toe.model' if os.path.exists('tic-tac-toe.model') else 'tic-tac-toe.model.keras'
loaded_model = tf.keras.models.load_model(path)

# Pick a few random rows from the test set
rng = np.random.default_rng()
idx = rng.choice(len(X_test), size=5, replace=False)
samples = X_test[idx]
actual = y_test[idx]

# Predict: softmax gives a probability per class, argmax picks the most likely one
probs = loaded_model.predict(samples, verbose=0)
predicted = np.argmax(probs, axis=1)

labels = {0: 'negative', 1: 'positive'}
for i in range(len(idx)):
    status = 'CORRECT' if predicted[i] == actual[i] else 'WRONG'
    print(f"Row {idx[i]:>3}: predicted = {labels[predicted[i]]:<8} "
          f"(confidence {probs[i].max():.2%}) | actual = {labels[actual[i]]:<8} -> {status}")

print(f"\n{(predicted == actual).sum()} / {len(idx)} predictions correct")

Row  99: predicted = negative (confidence 99.98%) | actual = negative -> CORRECT
Row  14: predicted = positive (confidence 100.00%) | actual = positive -> CORRECT
Row   6: predicted = negative (confidence 98.56%) | actual = negative -> CORRECT
Row  54: predicted = positive (confidence 100.00%) | actual = positive -> CORRECT
Row 103: predicted = positive (confidence 99.96%) | actual = positive -> CORRECT

5 / 5 predictions correct


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [7]:
import pandas as pd
import tensorflow as tf

def build_model(hidden_layers, learning_rate):
    """hidden_layers: list of unit counts, e.g. [128, 64]"""
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Input(shape=(X_train.shape[1],)))
    for units in hidden_layers:
        model.add(tf.keras.layers.Dense(units, activation='relu'))
    model.add(tf.keras.layers.Dense(2, activation='softmax'))

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)   # custom learning rate
    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

experiments = [
    # name,                    hidden layers,          learning rate, epochs
    ("Baseline",               [128, 128, 64],         0.001,         50),
    # --- number of layers ---
    ("Fewer layers",           [64],                   0.001,         50),
    ("More layers",            [256, 128, 128, 64, 32],0.001,         50),
    # --- learning rate ---
    ("Lower LR",               [128, 128, 64],         0.0001,        50),
    ("Higher LR",              [128, 128, 64],         0.01,          50),
    # --- epochs ---
    ("Fewer epochs",           [128, 128, 64],         0.001,         20),
    ("More epochs",            [128, 128, 64],         0.001,         150),
]

results = []
for name, layers, lr, epochs in experiments:
    tf.keras.utils.set_random_seed(42)          # same starting point for a fair comparison
    m = build_model(layers, lr)
    m.fit(X_train, y_train, epochs=epochs, batch_size=32, verbose=0)
    loss, acc = m.evaluate(X_test, y_test, verbose=0)
    results.append({"experiment": name, "layers": len(layers), "learning_rate": lr,
                    "epochs": epochs, "test_loss": round(loss, 4), "test_accuracy": round(acc, 4)})
    print(f"{name:<14} loss={loss:.4f}  acc={acc:.4f}")

results_df = pd.DataFrame(results).sort_values("test_loss")
print(results_df.to_string(index=False))

# Retrain the best configuration and save it as the improved model
best = experiments[[e[0] for e in experiments].index(results_df.iloc[0]["experiment"])]
tf.keras.utils.set_random_seed(42)
best_model = build_model(best[1], best[2])
best_model.fit(X_train, y_train, epochs=best[3], batch_size=32, verbose=0)
print("Best:", best[0], best_model.evaluate(X_test, y_test, verbose=0))

Baseline       loss=0.0737  acc=0.9844
Fewer layers   loss=0.2923  acc=0.9323
More layers    loss=0.0862  acc=0.9740
Lower LR       loss=0.1719  acc=0.9635
Higher LR      loss=0.1489  acc=0.9844
Fewer epochs   loss=0.0778  acc=0.9792
More epochs    loss=0.0886  acc=0.9844
  experiment  layers  learning_rate  epochs  test_loss  test_accuracy
    Baseline       3         0.0010      50     0.0737         0.9844
Fewer epochs       3         0.0010      20     0.0778         0.9792
 More layers       5         0.0010      50     0.0862         0.9740
 More epochs       3         0.0010     150     0.0886         0.9844
   Higher LR       3         0.0100      50     0.1489         0.9844
    Lower LR       3         0.0001      50     0.1719         0.9635
Fewer layers       1         0.0010      50     0.2923         0.9323
Best: Baseline [0.07370058447122574, 0.984375]


**Which approach(es) did you find helpful to improve your model performance?**

In [8]:
# Changing one thing at a time, compared with the baseline (3 hidden layers, lr = 0.001, 50 epochs):
#
# - Epochs: helped the most. With only 20 epochs the model was still learning
#   (higher loss). Going up to ~150 epochs pushed loss below 0.1 and accuracy above
#   0.95. Past a certain point, more epochs stopped helping, because the performance
#   had reached its ceiling.
#
# - Learning rate: helped. A very low rate (0.0001) learned too slowly to converge in
#   the same number of epochs. A higher rate (0.01) reached low loss faster, but can
#   be less stable from run to run.
#
# - More layers: helped a little compared with 1 hidden layer, but going from 3 to 5
#   layers made almost no difference. The tic-tac-toe data is not complex enough to
#   need a very deep network, so 2–3 hidden layers are enough.
#
# Conclusion: training longer and tuning the learning rate improved the model most,
# while adding layers beyond 2–3 did not.